In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
from collections import defaultdict
from tqdm import tqdm

from sklearn.metrics import silhouette_score

from time_series.data_generators import LorenzGenerator
from time_series.models import KernelRidgeRegression, RascuttiModel
from time_series.evaluators import MeanSquaredError

from experiment_logging import Experiment
from time_series_clustering import TimeSeriesClustering
from dataset_creator import create_dataset
from time_series.data_handlers import TimeSeriesData

import optuna

2026-01-08 13:37:05.538 | INFO     | time_series.config:<module>:13 - PROJ_ROOT path is: /home/james/Repo/PhD Repo/time_series_clustering
/home/james/Repo/PhD Repo/time_series_clustering/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Sweep noise

In [ ]:
experiment = Experiment("KRR - Noise Sweep", "experiments")

n_reps = 10
n_points = 400
n_correlated_dim = 5
n_uncorrelated_dim = 5

In [ ]:
noise_sweep = np.linspace(0, 2, 10)

for noise in tqdm(noise_sweep):
    datasets_params = [
        dict(
            theta=theta,
            n_points=n_points,
            n_correlated_dimensions=n_correlated_dim,
            n_uncorrelated_dimensions=n_uncorrelated_dim,
            noise=noise
        )
        for theta in np.linspace(0.1*np.pi, 0.5*np.pi, 20)
    ]

    datasets = [create_dataset(**params) for params in datasets_params]

    prepared_data = [TimeSeriesData(
            X=data[:-1],
            y=data[1:],
            train_val_test_split=[0.5, 0.4, 0.1],
            lag=1
        )
        for data in datasets
    ]

    def objective(trial):
        bandwidth = trial.suggest_float("bandwidth", 0.1, 4)
        reg = trial.suggest_float("reg", 1e-9, 1e-2)

        mse = 0
        for data in prepared_data:
            X_train, y_train = data.train_data()
            X_test, y_test = data.test_data()

            model = KernelRidgeRegression(kernel="rbf", bandwidth=bandwidth, reg=reg)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            mse += np.mean((y_pred - y_test)**2)

        return mse
            
    study = optuna.create_study()
    study.optimize(
        objective,
        n_trials=50,
        n_jobs=-1
    )

    best_params = study.best_params
    bandwidth = best_params["bandwidth"]
    reg = best_params["reg"]

    for rep in range(n_reps):
        ref_dataset = create_dataset(
            theta=0.3*np.pi,
            n_points=n_points,
            n_correlated_dimensions=n_correlated_dim,
            n_uncorrelated_dimensions=n_uncorrelated_dim,
            noise=noise
        )
